In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
from ptflops import get_model_complexity_info
import matplotlib.pyplot as plt

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model.config.forced_decoder_ids = None

In [ ]:
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

In [ ]:
sample = ds[0]["audio"]
input_features = processor(sample["array"], sampling_rate=sample["sampling_rate"], return_tensors="pt").input_features 
predicted_ids = model.generate(input_features)
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
transcription[0]

In [ ]:
import torch.nn as nn
class Decorator(nn.Module):
    def __init__(self, model):
        super(Decorator, self).__init__()
        self.model = model
    def forward(self, input_tensor):
        return self.model.generate(input_tensor)

In [ ]:
dummy_shape = (80, 3000)
flops = list()
sizes = sizes = ['tiny', 'base', 'small', 'medium']
for s in sizes:
    model = WhisperForConditionalGeneration.from_pretrained(f"openai/whisper-{s}")
    model_decorated = Decorator(model)
    macs, params = get_model_complexity_info(
        model_decorated,
        dummy_shape,
        as_strings=False,
        print_per_layer_stat=False,
        verbose=False,
    )
    flops.append(macs * 2 / 1e9)
flops

In [ ]:
seconds = range(1, 301)

In [ ]:
def sec2flops(s, flops):
    return (s // 30 + 1) * flops

tiny = [sec2flops(s, flops[0]) for s in seconds]
base = [sec2flops(s, flops[1]) for s in seconds]
small = [sec2flops(s, flops[2]) for s in seconds]
medium = [sec2flops(s, flops[3]) for s in seconds]

plt.figure(figsize=(8, 4))
plt.plot(seconds, tiny, label='Tiny', color='blue')
plt.plot(seconds, base, label='Base', color='orange')   
plt.plot(seconds, small, label='Small', color='green')
plt.plot(seconds, medium, label='Medium', color='red')
plt.xlabel('Seconds')
plt.ylabel('GFLOPs')
plt.title('GFLOPs vs Seconds for Whisper Models')
plt.legend()
plt.grid(True)
plt.show()
plt.close()

In [ ]:
def sec2size_audio(s, sample_rate=16000, bit_depth=16, channels=1):
    bytes_per_sample = bit_depth // 8
    num_samples = int(s * sample_rate * channels)
    data_bytes = num_samples * bytes_per_sample
    return data_bytes

def sec2size_text(audio_seconds, words_per_sec=2.5, bytes_per_word=6):
    return audio_seconds * words_per_sec * bytes_per_word

bytes_total = [sec2size_audio(s) + sec2size_text(s) for s in seconds]

plt.figure(figsize=(8, 4))
plt.plot(seconds, bytes_total, label='Total Size', color='purple')
plt.xlabel('Seconds')
plt.ylabel('Bytes')
plt.title('Total Size vs Seconds')
plt.legend()
plt.grid(True)
plt.show()
plt.close()